# Deteksi dan Pengenalan Pelat Nomor Kendaraan — Tahap 1: dataset dan preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-uji-t1/blob/main/notebooks/01_dataset_preprocessing.ipynb)

Kelompok `pcd2026-uji-t1` · 3 anggota · minimal **150 citra** (termasuk 45 foto sendiri).

Notebook ini membaca dataset dari folder Google Drive kelompok, memeriksa anotasi, menerapkan preprocessing, dan menyimpan gambar sebelum–sesudah. Jalankan dari atas ke bawah.

In [1]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-uji-t1"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-uji-t1")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']
JUMLAH_ANGGOTA = 3
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Citra   : /tmp/tmph7m3azk2/repo/data/sample
Anotasi : /tmp/tmph7m3azk2/repo/data/anotasi.csv (ada)


## 1. Statistik dataset

In [2]:
berkas = util.daftar_citra(FOLDER_CITRA)
if not berkas:
    raise RuntimeError(f"Tidak ada citra di {FOLDER_CITRA}")
stat = util.statistik_dataset(berkas)
print(f"{stat['jumlah']} citra ({stat['terbaca']} terbaca); minimal 150 untuk kelompok ini")
print("Lebar :", stat.get("lebar"))
print("Tinggi:", stat.get("tinggi"))
if stat["gagal"]:
    print("Gagal dibaca (hapus atau ganti):", *stat["gagal"][:10], sep="\n  ")
util.tampil(*[util.baca_citra(p) for p in berkas[:8]], judul=[p.name for p in berkas[:8]], kolom=4)

6 citra (6 terbaca); minimal 150 untuk kelompok ini
Lebar : {'min': 320, 'median': 320, 'maks': 320}
Tinggi: {'min': 240, 'median': 240, 'maks': 240}


## 2. Cek anotasi

Kotak lokasi pelat (x, y, lebar, tinggi dalam piksel) dan teks pelat tanpa spasi.

Kolom `anotasi.csv` (persis): `berkas, x, y, w, h, teks_pelat`. Contoh baris ada di `data/format_anotasi.csv`.

In [3]:
KOLOM_BERKAS = ['berkas']
if BERKAS_ANOTASI.exists():
    anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
    cek = util.cek_anotasi(anotasi, FOLDER_CITRA, KOLOM_BERKAS)
    print(len(anotasi), "baris anotasi")
    print("Disebut di anotasi tetapi berkasnya tidak ada:", cek["hilang"][:20])
    print("Citra belum beranotasi:", len(cek["tanpa_anotasi"]), cek["tanpa_anotasi"][:10])
else:
    print("anotasi.csv belum ada di", BERKAS_ANOTASI)

6 baris anotasi
Disebut di anotasi tetapi berkasnya tidak ada: []
Citra belum beranotasi: 0 []


## 3. Preprocessing

Saran untuk topik ini: grayscale; resize seragam (mis. sisi terpanjang 1024 px); peningkatan kontras (CLAHE) atau penajaman tepi.

Sel kode di bawah hanya **kerangka dummy**. Ganti dengan langkah preprocessing kelompok, lalu tulis **alasan setiap langkah** dan parameternya di sel markdown sesudahnya.

In [4]:
def praproses(citra):
    """DUMMY — ganti dengan langkah preprocessing kelompok Anda.

    Saran untuk topik ini ada di sel markdown di atas. Saat ini fungsi hanya memperkecil ukuran
    sehingga gambar sebelum–sesudah masih identik.
    """
    return util.ubah_ukuran_maks(citra, 1024)

**Alasan langkah preprocessing:** [isi: jelaskan setiap langkah dan parameternya]

## 4. Gambar sebelum–sesudah

Sel ini menyimpan 12 contoh ke folder `figures/` di Drive. Unggah minimal 10 ke `outputs/figures/` di repo (lihat `PANDUAN.md`).

In [5]:
FOLDER_FIGUR.mkdir(parents=True, exist_ok=True)
for p in berkas[:12]:
    asli = util.ubah_ukuran_maks(util.baca_citra(p), 1024)
    util.simpan_sebelum_sesudah(asli, praproses(asli), FOLDER_FIGUR / f"pra_{p.stem}.png")
contoh = sorted(FOLDER_FIGUR.glob("pra_*.png"))[:3]
print(len(list(FOLDER_FIGUR.glob("pra_*.png"))), "gambar di", FOLDER_FIGUR)
util.tampil(*[util.baca_citra(f) for f in contoh], kolom=1, ukuran=7)

6 gambar di /tmp/tmph7m3azk2/repo/data/figures


## Simpan ke GitHub

1. **Runtime → Restart session and run all**, pastikan tidak ada error.
2. **File → Save a copy in GitHub** → repo `Prodi-TI-IBIKesatuan/pcd2026-uji-t1` → **branch Anda** (bukan `main`) → path tetap `notebooks/01_dataset_preprocessing.ipynb`.
3. Buka *pull request* ke `main` di github.com. Langkah lengkap: `PANDUAN.md`.